# Perceptroni dhe ura drejt rrjetave nervore

**Lënda:** Machine Learning in Physics  
**Mjedisi:** Python, NumPy, pandas, Matplotlib dhe scikit-learn  
**Roli i notebook-ut:** mjeti kryesor i punës gjatë orës; leksioni i shkruar shërben për thellim teorik.

Perceptroni është fundi natyror i këtij kursi sepse mbledh disa ide të trajtuara më parë: klasifikim linear, funksion vendimi, përditësim iterativ dhe rëndësinë e gjeometrisë së të dhënave. Ai është një model i thjeshtë, por kufizimi i tij te problemet linearisht të ndashme na çon drejtpërdrejt te arsyeja pse nevojiten transformime jolineare dhe shtresa të fshehta.

Për këtë temë do t'u përgjigjemi sidomos tri pyetjeve:

1. Çfarë ndryshon në peshat kur një pikë klasifikohet gabim?
2. Kur garanton konvergjencë teorema e perceptronit?
3. Pse XOR nuk mund të zgjidhet nga një hiperplan i vetëm?

Ritmi i punës është: **parashiko → formulo matematikisht → implemento → ekzekuto → mat → interpreto → provo ta prishësh modelin**. Qelizat nuk duhen ekzekutuar mekanikisht njëra pas tjetrës. Para çdo eksperimenti duhet të kemi një pritshmëri dhe pas tij duhet të shpjegojmë mekanizmin që prodhoi rezultatin.


## Organizimi i seancës

| Koha | Veprimtaria | Rezultati i pritshëm |
|---:|---|---|
| 0–10 min | Pyetja shkencore dhe formulimi matematikor | Kuptimi i objektit që po modelojmë |
| 10–25 min | Zbatimi nga parimet e para | Lidhja ndërmjet formulës dhe algoritmit |
| 25–40 min | Eksperimenti numerik dhe figurat diagnostike | Kontrolli i sjelljes së modelit |
| 40–50 min | Krahasimi me implementimin profesional | Verifikimi i zbatimit dhe praktikës së mirë |
| 50–60 min | Interpretimi, kufizimet dhe një rast dështimi | Përfundim metodologjik, jo vetëm numerik |
| Pas orës | Punishtja dhe ushtrimet | Ndrysho një parametër, mat efektin dhe shpjego mekanizmin |

Kjo tabelë është orientuese. Nëse një diskutim fizik ose matematik është i frytshëm, është më e vlefshme të ndalemi te ai sesa të “mbarojmë” notebook-un.


## Rezultatet e të nxënit

Në fund të punës studenti duhet të jetë në gjendje të:

- Shkruaj weighted sum dhe pragun.
- Zbatoj rregullin e përditësimit.
- Interpretoj historinë e gabimeve.
- Shpjegoj kushtin e konvergjencës.
- Vizualizoj dhe shpjegoj kufizimin XOR.


### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Parashikim para ekzekutimit.** Nëse të dhënat janë linearisht të ndashme, numri i gabimeve duhet të shkojë në zero; nëse nuk janë, prisni oscilim ose gabime të mbetura.


In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("12 — Perceptroni dhe ura drejt rrjetave nervore")
print("Seed:", RANDOM_STATE)

**Interpretim.** Konvergjenca ose mungesa e saj lidhet me gjeometrinë e klasave. Dështimi në një dataset jo të ndashëm linearisht nuk është domosdoshmërisht gabim implementimi.


# 1. Neuroni artificial si ndarës linear — CORE

Perceptroni përdor
$$
z=\mathbf w^T\mathbf x+b,
$$
dhe vendimin
$$
\hat y=\mathrm{sign}(z).
$$
Nëse një mostër $(x_i,y_i)$ klasifikohet gabim, përditësojmë
$$
\mathbf w\leftarrow\mathbf w+\eta y_i\mathbf x_i,\qquad
b\leftarrow b+\eta y_i.
$$

### Para se të ekzekutojmë: Klasat linearisht të ndashme

Fillojmë me një rast ku teorema e konvergjencës së perceptronit ka kuptim. Nëse klasat ndahen me një hiperplan, algoritmi duhet të gjejë një zgjidhje pas një numri të fundëm gabimesh.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Klasat linearisht të ndashme.
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X0=rng.normal([-1,-1],[0.55,0.55],size=(90,2))
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X1=rng.normal([1.2,1.0],[0.55,0.55],size=(90,2))
X=np.vstack([X0,X1]); y=np.r_[np.zeros(len(X0)),np.ones(len(X1))]
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(X0[:,0],X0[:,1],s=14,label="klasa 0")
ax.scatter(X1[:,0],X1[:,1],s=14,label="klasa 1")
ax.legend(); ax.set_title("Të dhëna linearisht të ndashme")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 2. Zbatim nga parimet e para — CORE

### Para se të ekzekutojmë: Rregulli i mësimit nga gabimi

Implementimi manual është i shkurtër, por çdo hap është konceptualisht i rëndësishëm: transformojmë etiketat në $\pm1$, kontrollojmë margjinën e shenjës dhe ndryshojmë peshat vetëm kur mostra nuk klasifikohet saktë.

**Parashikim para ekzekutimit.** Para se të vizatoni historinë e humbjes, vendosni çfarë prisni për learning rate të vogël, të përshtatshëm dhe tepër të madh.


In [ ]:
# Qëllimi i këtij blloku: Rregulli i mësimit nga gabimi.
def train_perceptron(X,y,eta=1.0,max_epochs=100,seed=42):
    yt=np.where(y>0,1.0,-1.0)
    w=np.zeros(X.shape[1]); b=0.0
    rng_local=np.random.default_rng(seed)
    mistakes=[]
    # Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
    for _ in range(max_epochs):
        m=0
        for i in rng_local.permutation(len(X)):
            if yt[i]*(X[i]@w+b)<=0:
                w+=eta*yt[i]*X[i]
                b+=eta*yt[i]
                m+=1
        mistakes.append(m)
        if m==0: break
    return w,b,np.array(mistakes)

w,b,mist=train_perceptron(X,y)
w,b,len(mist)

### Perceptroni kanonik nga `src/`

### Para se të ekzekutojmë: Versioni kanonik i perceptronit

Pas implementimit didaktik përdorim klasën e `src/`. Kjo siguron një version të ripërdorshëm pa humbur transparencën matematikore të hapit të mëparshëm.

**Parashikim para ekzekutimit.** Nëse të dhënat janë linearisht të ndashme, numri i gabimeve duhet të shkojë në zero; nëse nuk janë, prisni oscilim ose gabime të mbetura.


In [ ]:
# Qëllimi i këtij blloku: Versioni kanonik i perceptronit.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.perceptron.perceptron import PerceptronFromScratch
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
per_src = PerceptronFromScratch(max_epochs=100, seed=RANDOM_STATE).fit(X, y)
print("Epokat:", per_src.n_iter_, "gabimet e fundit:", per_src.mistakes_per_epoch_[-1])

**Interpretim.** Konvergjenca ose mungesa e saj lidhet me gjeometrinë e klasave. Dështimi në një dataset jo të ndashëm linearisht nuk është domosdoshmërisht gabim implementimi.


# 3. Kufiri dhe konvergjenca — CORE

### Para se të ekzekutojmë: Kufiri dhe historia e gabimeve

Vizatojmë njëkohësisht hiperplanin final dhe numrin e gabimeve për epokë. Konvergjenca nuk është thjesht ulje e një loss-i të vazhdueshëm; te perceptroni ajo shfaqet si epokë pa gabime.

**Parashikim para ekzekutimit.** Nëse të dhënat janë linearisht të ndashme, numri i gabimeve duhet të shkojë në zero; nëse nuk janë, prisni oscilim ose gabime të mbetura.


In [ ]:
# Qëllimi i këtij blloku: Kufiri dhe historia e gabimeve.
xx=np.linspace(X[:,0].min()-1,X[:,0].max()+1,200)
yy=-(w[0]*xx+b)/w[1]
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],c=y,s=14)
ax.plot(xx,yy)
ax.set_title("Kufiri i perceptronit")
plt.show()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(np.arange(1,len(mist)+1),mist,marker="o")
ax.set_xlabel("Epoka"); ax.set_ylabel("Gabimet")
ax.set_title("Konvergjenca")
plt.show()

**Interpretim.** Konvergjenca ose mungesa e saj lidhet me gjeometrinë e klasave. Dështimi në një dataset jo të ndashëm linearisht nuk është domosdoshmërisht gabim implementimi.


# 4. Krahasim me sklearn — CORE

### Para se të ekzekutojmë: Krahasimi me implementimin profesional

Koeficientët nuk kanë pse të jenë identikë sepse perceptroni mund të konvergojë në disa kufij ndarës. Ajo që duhet të krahasojmë është sjellja e klasifikimit dhe konsistenca e parimit.

**Parashikim para ekzekutimit.** Nëse të dhënat janë linearisht të ndashme, numri i gabimeve duhet të shkojë në zero; nëse nuk janë, prisni oscilim ose gabime të mbetura.


In [ ]:
# Qëllimi i këtij blloku: Krahasimi me implementimin profesional.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.linear_model import Perceptron
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
p=Perceptron(random_state=RANDOM_STATE,max_iter=1000,tol=1e-4).fit(X,y)
print("manual:",w,b)
print("sklearn:",p.coef_[0],p.intercept_[0])

**Interpretim.** Konvergjenca ose mungesa e saj lidhet me gjeometrinë e klasave. Dështimi në një dataset jo të ndashëm linearisht nuk është domosdoshmërisht gabim implementimi.


# 5. Të dhëna me zhurmë dhe jo të ndashme — CORE

### Para se të ekzekutojmë: Kur hipoteza e ndarjes lineare prishet

Përmbysim disa etiketa që të krijojmë konflikt të brendshëm. Në këtë rast algoritmi vazhdon të bëjë gabime sepse nuk ekziston asnjë hiperplan që i kënaq të gjitha mostrat.

**Parashikim para ekzekutimit.** Nëse të dhënat janë linearisht të ndashme, numri i gabimeve duhet të shkojë në zero; nëse nuk janë, prisni oscilim ose gabime të mbetura.


In [ ]:
# Qëllimi i këtij blloku: Kur hipoteza e ndarjes lineare prishet.
Xn=X.copy()
yn=y.copy()
flip=rng.choice(len(yn),size=22,replace=False)
yn[flip]=1-yn[flip]
_,_,mist_n=train_perceptron(Xn,yn,max_epochs=60)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(mist_n)
ax.set_xlabel("Epoka"); ax.set_ylabel("Gabimet")
ax.set_title("Perceptroni nuk konvergon kur ndarja lineare është e pamundur")
plt.show()

**Interpretim.** Konvergjenca ose mungesa e saj lidhet me gjeometrinë e klasave. Dështimi në një dataset jo të ndashëm linearisht nuk është domosdoshmërisht gabim implementimi.


# 6. XOR: kufiri themelor — CORE

### Para se të ekzekutojmë: XOR si kundërshembull strukturor

XOR është shembulli klasik ku të katër pikat janë krejt të pastra, por jo linearisht të ndashme. Dështimi nuk vjen nga zhurma apo optimizimi: klasa e modelit është e pamjaftueshme.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: XOR si kundërshembull strukturor.
Xxor=np.array([[0,0],[0,1],[1,0],[1,1]],float)
yxor=np.array([0,1,1,0])
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(Xxor[:,0],Xxor[:,1],c=yxor,s=140)
for i,(a,bx) in enumerate(Xxor):
    ax.text(a+0.03,bx+0.03,str(yxor[i]))
ax.set_xlim(-0.3,1.3); ax.set_ylim(-0.3,1.3)
ax.set_title("XOR nuk ndahet me një vijë të vetme")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


**Interpretim.** Dështimi në XOR nuk është problem i algoritmit të trajnimit; është kufizim i klasës së modeleve lineare. Kjo motivon shtresa të fshehta dhe transformime jolineare.

Në një analizë reale ky interpretim duhet të shoqërohet me kontroll të njësive, madhësive karakteristike dhe hipotezave që lejojnë kalimin nga një tregues numerik te një përfundim fizik.

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Ura drejt rrjetave nervore — EXTENDED

Një rrjet me shumë shtresa kombinon transformime lineare me funksione aktivizimi jolineare:
$$
h=\phi(Wx+b).
$$
Shtresat e fshehta mund të ndërtojnë përfaqësime ku probleme si XOR bëhen linearisht të ndashme.
Trajnimi i rrjeteve moderne përdor **prapapërhapjen (backpropagation)** dhe variante të zbritjes
stokastike të gradientit. Këtu ndalemi: këto tema i përkasin trajtimit të mëvonshëm në nivel Master.

### Para se të ekzekutojmë: Një përfaqësim i ri ndryshon problemin

Ndërtojmë me dorë një veçori jolineare që e bën XOR të thjeshtë. Ky eksperiment tregon idenë themelore pas shtresave të fshehta: modeli mëson ose ndërton një përfaqësim ku ndarja bëhet më e lehtë.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Një përfaqësim i ri ndryshon problemin.
# Një transformim i thjeshtë i projektuar me dorë që e bën XOR të dallueshëm
z = (Xxor[:,0] != Xxor[:,1]).astype(float)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(z, np.zeros_like(z), c=yxor, s=140)
ax.set_xlabel("veçori e re jolineare: x1 XOR x2")
ax.set_yticks([])
ax.set_title("Një përfaqësim jolinear mund ta zgjidhë problemin")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 8. Aplikim fizik: ndarje regjimesh — EXTENDED

Në një hapësirë veçorish të ndërtuar nga amplituda dhe frekuenca, perceptroni mund të funksionojë mirë vetëm nëse regjimet ndahen afërsisht me një hiperplan. Ky kusht duhet kontrolluar, jo supozuar.

## Punishtja — ndrysho, mat dhe shpjego

Në këtë pjesë nuk kërkohet ekzekutim pasiv. Për çdo eksperiment:

1. shkruani paraprakisht çfarë prisni të ndodhë;
2. ndryshoni **vetëm një** element në të njëjtën kohë;
3. ruani metrikën ose figurën që ndryshon;
4. shpjegoni mekanizmin matematik ose statistikor;
5. dalloni qartë atë që tregon eksperimenti nga ajo që **nuk** mund të përfundohet prej tij.

### Pyetje për diskutim

1. Si ndryshon perceptroni nga regresioni logjistik?
2. Çfarë do të thotë linear separability?
3. Pse zhurma e etiketave pengon konvergjencën?
4. Cila ide e perceptronit mbijeton te rrjetat moderne dhe cila ndryshon?


# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Ndryshoni learning rate dhe numëroni epokat deri në konvergjencë.
2. Zhvendosni klasat gradualisht derisa të mbivendosen.
3. Krahasoni perceptronin me regresionin logjistik.
4. Shpjegoni pse XOR kërkon jolinearitet.

        ## B. Ushtrim i plotë
        Ndërtoni një dataset me tre veçori ku klasat janë të ndashme dhe verifikoni konvergjencën e perceptronit; më pas shtoni zhurmë etikete dhe analizoni gabimet për epokë.

        ## C. Playground
        Ndryshoni learning rate, seed, mbivendosjen dhe numrin e gabimeve në etiketa.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Perceptroni dhe ura drejt rrjetave nervore** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.

## Kontrolli përfundimtar

- [ ] Shkruaj weighted sum dhe pragun.
- [ ] Zbatoj rregullin e përditësimit.
- [ ] Interpretoj historinë e gabimeve.
- [ ] Shpjegoj kushtin e konvergjencës.
- [ ] Vizualizoj dhe shpjegoj kufizimin XOR.
